# Week 4 Advanced — Cascaded Loops, Bandwidth Separation, and Sampling

Now we treat torque as a controlled actuator rather than an ideal input. The outer speed loop commands torque; the inner loop tracks that torque with finite bandwidth, saturation, and rate limits.

Outer plant: $$J\dot{\omega}=T_{act}-T_L-b\omega$$
Inner actuator: $$\tau_T\dot{T}_{act}=T^* - T_{act}.$$

The engineering question is: **how much faster must the inner loop be before the cascade behaves like the outer-loop design assumes?**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

J,b=0.08,0.03
Tmax,rate=3.0,120.0
dt=5e-4
t=np.arange(0,5,dt)


## 1. Sweep inner-loop bandwidth

Keep the outer PI gains fixed. Change only the actuator time constant. This isolates the effect of bandwidth separation.

In [ ]:
def run(tau_inner):
    omega=0.; torque=0.; integ=0.
    W=[]; TC=[]; TA=[]
    for ti in t:
        ref=30.0
        load=0.4 if ti<2.5 else 1.0
        e=ref-omega
        integ += e*dt
        tc=np.clip(0.28*e+1.2*integ,-Tmax,Tmax)
        dT=np.clip((tc-torque)/tau_inner,-rate,rate)
        torque += dT*dt
        omega += (torque-load-b*omega)/J*dt
        W.append(omega); TC.append(tc); TA.append(torque)
    return np.array(W),np.array(TC),np.array(TA)

for tau in [0.005,0.02,0.08,0.2]:
    W,_,_=run(tau)
    plt.plot(t,W,label=f'tau={tau}s')
plt.axvline(2.5,ls=':'); plt.xlabel('Time [s]'); plt.ylabel('Speed [rad/s]'); plt.grid(True); plt.legend(); plt.show()


## 2. Sampling is part of the controller

For a continuous plant $\dot{x}=Ax+Bu$, exact zero-order-hold discretization gives
$$x_{k+1}=A_dx_k+B_du_k,$$
$$A_d=e^{AT_s},\qquad B_d=\int_0^{T_s}e^{A\tau}B\,d\tau.$$

A stable continuous controller can still become poor or unstable if the sample time is too slow.

In [ ]:
def sampled_pi(Ts):
    omega=0.; integ=0.; u=0.
    dtp=2e-4; tt=np.arange(0,4,dtp); next_sample=0.
    W=[]
    for ti in tt:
        if ti+1e-12 >= next_sample:
            e=30.-omega
            integ += e*Ts
            u=np.clip(0.28*e+1.2*integ,-Tmax,Tmax)
            next_sample += Ts
        load=0.5 if ti<2 else 1.0
        omega += (u-load-b*omega)/J*dtp
        W.append(omega)
    return tt,np.array(W)

for Ts in [0.001,0.01,0.05,0.12]:
    tt,W=sampled_pi(Ts); plt.plot(tt,W,label=f'Ts={Ts}s')
plt.grid(True); plt.legend(); plt.xlabel('Time [s]'); plt.ylabel('Speed [rad/s]'); plt.show()


## Engineering tasks

- Define an acceptable outer-loop settling time and infer a target bandwidth.
- Find the slowest inner-loop time constant that still gives less than 10% degradation.
- Find the largest sample period that keeps the response acceptable.
- Add one-sample computational delay and repeat.
- Explain why a 20–40 kHz inverter current loop and a much slower supervisory loop should not share the same update rate.

**Deliverable:** justify inner-loop bandwidth and sample rate numerically instead of choosing them by habit.